1. Setup and Configuration
This section ensures all necessary libraries are installed and ready, sets the environment's configuration, and checks for a GPU.

Code Chunk 1: Imports and Global Setup

In [1]:
import pandas as pd
import numpy as np
import os
import re
from tqdm import tqdm
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments, logging
from datasets import Dataset

# Suppress Hugging Face warnings for cleaner output
logging.set_verbosity_error()

# Ensure NLTK resources are downloaded (required for text cleaning)
try:
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
except Exception:
    pass

c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Code Chunk 2: Configuration - File Path and Device Check

In [ ]:
# CODE CHUNK 2: Configuration - File Path and Device Check

# --- ⚠️ IMPORTANT: CHANGE THIS PATH TO YOUR CSV FILE LOCATION ---
data_path = r"Phishing_Email 4.csv"

# Check for GPU (CUDA) availability and set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n--- Running on device: {device} ---")


--- Running on device: cpu ---


2. Data Loading and Initial Preprocessing
This section loads the data, standardizes the column names, handles missing values, and maps the text labels to numerical indices.

Code Chunk 3: Data Loading and Target Mapping

In [3]:
# CODE CHUNK 3: Data Loading and Target Mapping

# 1. Load data and handle potential errors
try:
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"File missing. Check your path: {data_path}")
    
    df = pd.read_csv(data_path, low_memory=False)
    print(f"\n--- Dataset Loaded Successfully ({len(df)} total emails) ---")
    
except Exception as e:
    print(f"FATAL ERROR: Data loading failed: {e}")
    exit()

# 2. Standardize column names (lowercase, replace spaces with underscores)
df.columns = df.columns.str.lower().str.replace(' ', '_')

# 3. Rename columns and drop missing values
df = df.rename(columns={'email_text': 'text', 'email_type': 'target'}, errors='ignore')
df.dropna(subset=['text', 'target'], inplace=True)

# 4. Map text labels to numerical labels (0 and 1)
unique_targets = df['target'].unique()
if len(unique_targets) == 2:
    target_counts = df['target'].value_counts()
    minority_label = target_counts.index[target_counts.argmin()]
    majority_label = target_counts.index[target_counts.argmax()]
    # Map Majority (Safe) to 0 and Minority (Phishing) to 1
    target_mapping = {majority_label: 0, minority_label: 1} 
    df['label'] = df['target'].map(target_mapping).astype(int)
    print(f"--- Target mapping: {majority_label} -> 0 (Safe), {minority_label} -> 1 (Phishing) ---")
else:
    print("WARNING: Target column has issues. Found more/less than 2 unique values.")
    exit()


--- Dataset Loaded Successfully (3457 total emails) ---
--- Target mapping: Safe Email -> 0 (Safe), Phishing Email -> 1 (Phishing) ---


3. Text Cleaning and Data Splitting

Code Chunk 4: Text Cleaning Function

In [4]:
# CODE CHUNK 4: Text Cleaning Definition

def clean_text(text):
    """Performs basic text cleaning for standardization."""
    text = str(text).lower()
    # Remove HTML tags
    text = re.sub('<.*?>', '', text)
    # Remove all non-alphabetic characters and non-whitespace
    text = re.sub(r'[^a-z\s]', '', text)
    # Reduce multiple spaces to a single space and strip
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply the cleaning function to the dataset
tqdm.pandas(desc="Cleaning text")
df['cleaned_text'] = df['text'].progress_apply(clean_text)

Cleaning text: 100%|██████████| 3457/3457 [00:00<00:00, 3673.53it/s]


Code Chunk 5: Splitting Data into Training and Testing Sets

In [5]:
# CODE CHUNK 5: Splitting Data into Training and Testing Sets

X = df['cleaned_text'].values
y = df['label'].values

# Split data 80% for training and 20% for testing
# stratify=y ensures the same class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTrain samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")


Train samples: 2765
Test samples: 692


4. Baseline Models (Optional Comparison)
This optional section quickly trains and evaluates simple models for a baseline comparison to the more complex DistilBERT model.

Code Chunk 6: Training Traditional Baseline Models (TF-IDF)

In [6]:
# CODE CHUNK 6: Training Traditional Baseline Models (TF-IDF)

print("\n--- Training Traditional Baseline Models (TF-IDF) ---")

# 1. Calculate class weights for imbalance correction
class_labels = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=class_labels, y=y_train)
class_weight_dict = dict(zip(class_labels, class_weights))
print(f"Calculated Class Weights (used in LR): {class_weight_dict}") 

# 2. Feature Extraction: TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 3. Model 1: Weighted Logistic Regression
log_reg = LogisticRegression(class_weight=class_weight_dict, solver='liblinear', random_state=42, max_iter=1000)
log_reg.fit(X_train_tfidf, y_train)
y_pred_lr = log_reg.predict(X_test_tfidf)
print("\n--- Results: Logistic Regression (Weighted) ---")
print(classification_report(y_test, y_pred_lr, zero_division=0))

# 4. Model 2: Naive Bayes (Unweighted)
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)
print("\n--- Results: Naive Bayes (Unweighted) ---")
print(classification_report(y_test, y_pred_nb, zero_division=0))


--- Training Traditional Baseline Models (TF-IDF) ---
Calculated Class Weights (used in LR): {np.int64(0): np.float64(0.8363581367211131), np.int64(1): np.float64(1.2432553956834533)}

--- Results: Logistic Regression (Weighted) ---
              precision    recall  f1-score   support

           0       0.97      0.96      0.97       414
           1       0.95      0.96      0.95       278

    accuracy                           0.96       692
   macro avg       0.96      0.96      0.96       692
weighted avg       0.96      0.96      0.96       692


--- Results: Naive Bayes (Unweighted) ---
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       414
           1       0.96      0.93      0.94       278

    accuracy                           0.96       692
   macro avg       0.96      0.95      0.95       692
weighted avg       0.96      0.96      0.96       692



5. DistilBERT Fine-Tuning
This section sets up the Hugging Face Trainer environment, preparing the data for the Transformer model.

Code Chunk 7: Tokenization and Hugging Face Datasets

In [7]:
# CODE CHUNK 7: DistilBERT Tokenization and Hugging Face Datasets

# 1. Initialize the DistilBERT tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
MAX_LENGTH = 512

# 2. Define the tokenization function
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=MAX_LENGTH)

# 3. Create Hugging Face Dataset objects from the splits
train_df = pd.DataFrame({'text': X_train.tolist(), 'label': y_train.tolist()})
test_df = pd.DataFrame({'text': X_test.tolist(), 'label': y_test.tolist()})

train_dataset_raw = Dataset.from_pandas(train_df)
test_dataset_raw = Dataset.from_pandas(test_df)

# 4. Apply tokenization to both datasets
print("\n--- Starting Tokenization for Hugging Face Dataset ---")
tokenized_train_dataset = train_dataset_raw.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset_raw.map(tokenize_function, batched=True)

# 5. Format datasets for PyTorch Trainer
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["text"])
tokenized_test_dataset = tokenized_test_dataset.remove_columns(["text"])
tokenized_train_dataset.set_format("torch")
tokenized_test_dataset.set_format("torch")

print("--- Tokenization Complete and HF Datasets Created ---")


--- Starting Tokenization for Hugging Face Dataset ---


Map: 100%|██████████| 692/692 [00:00<00:00, 739.26 examples/s]

--- Tokenization Complete and HF Datasets Created ---


6. Model Training
Code Chunk 8: Load Model and Define Metrics

In [8]:
# CODE CHUNK 8: Load Model and Define Metrics

# 1. Load the pre-trained DistilBERT model for sequence classification (2 classes)
model_distilbert = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
).to(device)

# 2. Define the metric computation function
def compute_metrics(p):
    """Calculates accuracy, precision, recall, and F1-score."""
    predictions = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    # Prioritize F1 for the positive class (1: Phishing)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0, pos_label=1
    )
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

Code Chunk 9: Training Arguments and Execution

This would take a while at least 3,100mins depending on your PC Specs

In [9]:
# CODE CHUNK 9: Training Arguments and Execution

BATCH_SIZE = 16
N_EPOCHS = 3
RUN_NAME = "distilbert-phishing-detector-vscode"

# 1. Define Training Arguments
training_args = TrainingArguments(
    output_dir=f"./results/{RUN_NAME}",
    num_train_epochs=N_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch", # Evaluate at the end of each epoch
    save_strategy="epoch", # Save checkpoint at the end of each epoch
    load_best_model_at_end=True, # Load the best model after training finishes
    metric_for_best_model='f1', # Metric used to determine the 'best' model
)

# 2. Initialize the Trainer
trainer = Trainer(
    model=model_distilbert,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
)

# 3. Start Fine-Tuning
print(f"\n--- Starting DistilBERT Fine-Tuning (Epochs={N_EPOCHS}, Batch={BATCH_SIZE}, Device={device}) ---")
trainer.train()
print("--- DistilBERT Fine-Tuning Complete ---")


--- Starting DistilBERT Fine-Tuning (Epochs=3, Batch=16, Device=cpu) ---


c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.6004, 'grad_norm': 3.41795015335083, 'learning_rate': 9.900000000000002e-06, 'epoch': 0.5780346820809249}
{'eval_loss': 0.13114117085933685, 'eval_accuracy': 0.9523121387283237, 'eval_f1': 0.9398907103825137, 'eval_precision': 0.9520295202952029, 'eval_recall': 0.9280575539568345, 'eval_runtime': 653.0118, 'eval_samples_per_second': 1.06, 'eval_steps_per_second': 0.067, 'epoch': 1.0}


c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.1964, 'grad_norm': 1.8310203552246094, 'learning_rate': 1.9900000000000003e-05, 'epoch': 1.1560693641618498}
{'loss': 0.1227, 'grad_norm': 0.7457367181777954, 'learning_rate': 2.9900000000000002e-05, 'epoch': 1.7341040462427746}
{'eval_loss': 0.18760299682617188, 'eval_accuracy': 0.9479768786127167, 'eval_f1': 0.9389830508474576, 'eval_precision': 0.8878205128205128, 'eval_recall': 0.9964028776978417, 'eval_runtime': 791.9074, 'eval_samples_per_second': 0.874, 'eval_steps_per_second': 0.056, 'epoch': 2.0}


c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.1255, 'grad_norm': 15.003544807434082, 'learning_rate': 3.99e-05, 'epoch': 2.3121387283236996}
{'loss': 0.084, 'grad_norm': 1.8117187023162842, 'learning_rate': 4.99e-05, 'epoch': 2.8901734104046244}
{'eval_loss': 0.09139639139175415, 'eval_accuracy': 0.976878612716763, 'eval_f1': 0.9716312056737588, 'eval_precision': 0.958041958041958, 'eval_recall': 0.9856115107913669, 'eval_runtime': 734.0035, 'eval_samples_per_second': 0.943, 'eval_steps_per_second': 0.06, 'epoch': 3.0}
{'train_runtime': 187034.0518, 'train_samples_per_second': 0.044, 'train_steps_per_second': 0.003, 'train_loss': 0.21915344018237898, 'epoch': 3.0}
--- DistilBERT Fine-Tuning Complete ---


7. Evaluation and Model Saving

Code Chunk 10: Evaluation and Model Saving

In [12]:
# CODE CHUNK 10: Evaluation and Model Saving
#
# 1. Save the best fine-tuned model to a local directory
save_path = './distilbert_phishing_detector_pytorch'
trainer.save_model(save_path)
print(f"\nModel saved to: {os.path.abspath(save_path)}")

# 2. Final Evaluation
print("\n--- Final Evaluation of Best Model on Test Set ---")
evaluation_results = trainer.evaluate(tokenized_test_dataset)
print("\nEvaluation Summary:")
print(evaluation_results)

# 3. Generate detailed classification report
predictions_raw = trainer.predict(tokenized_test_dataset)
predictions = np.argmax(predictions_raw.predictions, axis=1)

print("\n--- Detailed Classification Report ---")
# The y_test is the ground truth labels from the original split
print(classification_report(y_test, predictions, zero_division=0))
print("\n*** Next Step: The saved model can now be used for inference or an XAI layer (LIME/SHAP). ***")


Model saved to: c:\Users\pc\AppData\Local\Programs\Microsoft VS Code\distilbert_phishing_detector_pytorch

--- Final Evaluation of Best Model on Test Set ---


c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.09139639139175415, 'eval_accuracy': 0.976878612716763, 'eval_f1': 0.9716312056737588, 'eval_precision': 0.958041958041958, 'eval_recall': 0.9856115107913669, 'eval_runtime': 738.9217, 'eval_samples_per_second': 0.936, 'eval_steps_per_second': 0.06, 'epoch': 3.0}

Evaluation Summary:
{'eval_loss': 0.09139639139175415, 'eval_accuracy': 0.976878612716763, 'eval_f1': 0.9716312056737588, 'eval_precision': 0.958041958041958, 'eval_recall': 0.9856115107913669, 'eval_runtime': 738.9217, 'eval_samples_per_second': 0.936, 'eval_steps_per_second': 0.06, 'epoch': 3.0}


c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



--- Detailed Classification Report ---
              precision    recall  f1-score   support

           0       0.99      0.97      0.98       414
           1       0.96      0.99      0.97       278

    accuracy                           0.98       692
   macro avg       0.97      0.98      0.98       692
weighted avg       0.98      0.98      0.98       692


*** Next Step: The saved model can now be used for inference or an XAI layer (LIME/SHAP). ***


8. Explainable AI (XAI) Analysis
This section imports necessary XAI libraries (like captum and lime) and uses the saved DistilBERT model to explain specific predictions.

Code Chunk 11: Setup and Prediction Function

In [ ]:
# CODE CHUNK 11: XAI Setup, Imports, and Prediction Function

# --- XAI Libraries (Need to be installed: pip install lime-explainer captum) ---
from lime.lime_text import LimeTextExplainer
from captum.attr import IntegratedGradients
from captum.attr import TokenwiseSaliency
from captum.attr import visualization as viz
import matplotlib.pyplot as plt

# 1. Load the saved model and tokenizer
# NOTE: Ensure the path matches the save_path from Code Chunk 10
save_path = './distilbert_phishing_detector_pytorch' 
model_xai = DistilBertForSequenceClassification.from_pretrained(save_path).to(device)
tokenizer_xai = DistilBertTokenizerFast.from_pretrained(save_path)
model_xai.eval() # Set model to evaluation mode

# 2. Define the prediction function required by LIME and Integrated Gradients
def predict_proba(texts):
    """
    Function required by LIME/Captum. Takes raw texts and returns 
    the probability scores (logits) for the two classes (Safe/Phishing).
    """
    inputs = tokenizer_xai(texts, return_tensors='pt', padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model_xai(**inputs)
    
    # Return the raw logits (scores) for the model's classes
    return outputs.logits.cpu().numpy()

print("\n--- XAI Setup Complete: Model and Tokenizer loaded successfully ---")

Code Chunk 12: LIME Analysis (Local Explanation)

In [ ]:
# CODE CHUNK 12: LIME (Local Interpretability) Analysis

# --- Example Email for Explanation ---
# NOTE: Replace this with an actual phishing email text from your test set (y_test=1)
sample_text = X_test[np.where(y_test == 1)[0][0]] # Take the first phishing email from test set
target_class_index = 1 # We want to explain the prediction for the Phishing class (1)

# 1. Initialize the LIME Explainer
# class_names should match your target labels (0=Safe, 1=Phishing)
explainer = LimeTextExplainer(class_names=['Safe Email', 'Phishing Email'])

# 2. Generate the explanation
print("\n--- 4. LIME (Local Interpretability) Analysis ---")
print(f"Explaining Prediction for: '{sample_text[:100]}...'")

explanation = explainer.explain_instance(
    sample_text,
    classifier_fn=predict_proba,
    num_features=10, # Show top 10 contributing words
    labels=(target_class_index,)
)

# 3. Print the results (tokens and their weights)
print(f"LIME Explanation for Class: {'Phishing Email'}")
for feature, weight in explanation.as_list(label=target_class_index):
    print(f"  Token: '{feature}' | Weight: {weight:.4f}")

# 4. Optional: Save the explanation as an HTML file
# explanation.save_to_file('./lime_explanation.html')

Code Chunk 13: Integrated Gradients (IG) Analysis

In [ ]:
# CODE CHUNK 13: Integrated Gradients (IG) Analysis

# 1. Prepare input for Captum (Requires tensors)
ig_inputs = tokenizer_xai(sample_text, return_tensors='pt', truncation=True, padding=True, max_length=512)
input_ids = ig_inputs['input_ids'].to(device)
attention_mask = ig_inputs['attention_mask'].to(device)

# 2. Get the word embedding layer (required for IG)
# This finds the layer that converts input IDs to numerical vectors
def forward_func(input_ids, attention_mask=None):
    return model_xai(input_ids, attention_mask=attention_mask).logits

# 3. Initialize Integrated Gradients
ig = IntegratedGradients(forward_func)

# 4. Calculate IG scores (Attributions)
print("\n--- 5. Integrated Gradients (IG) Analysis ---")
print("Calculating Integrated Gradients (Targeting Embeddings)...")

attributions_ig, delta = ig.attribute(
    input_ids, 
    target=target_class_index, # Explain the Phishing class (1)
    return_convergence_delta=True
)

# 5. Token-wise saliency (map IG scores back to tokens)
token_saliency = TokenwiseSaliency(forward_func, tokenizer_xai)
attributions = token_saliency.attribute(input_ids, target=target_class_index)
attributions_sum = torch.sum(attributions, dim=1) # Sum of attributions for each token

# 6. Get token labels
all_tokens = tokenizer_xai.convert_ids_to_tokens(input_ids[0].tolist())

# 7. Print the top contributing tokens based on absolute IG score
print(f"Integrated Gradients Explanation for Class: {'Phishing Email'}")
# Create a DataFrame to easily sort the tokens by impact
results_df = pd.DataFrame({
    'Token': all_tokens,
    'IG Score (Logit)': attributions_sum.cpu().numpy()
})
results_df['Absolute Score'] = results_df['IG Score (Logit)'].abs()
top_tokens = results_df.nlargest(10, 'Absolute Score').drop(columns='Absolute Score')

print("Top 10 tokens ranked by absolute IG score (Impact):")
print(top_tokens.to_string(index=False))

9. Integrated Gradients Visualization (Heatmap)
This final chunk uses the captum library to generate a visual heatmap, making it easy to see which words most influenced the model's prediction.

Code Chunk 14: Integrated Gradients Visualization

In [ ]:
# CODE CHUNK 14: Integrated Gradients Visualization (HEATMAP)

# NOTE: This chunk requires running the previous chunks to have 'attributions_sum', 'input_ids', and 'all_tokens'

# 1. Prepare data for visualization
# Convert input IDs and attribution scores to a list/array format
token_ids = input_ids[0].cpu().numpy().tolist()
word_attributions = attributions_sum.cpu().numpy().tolist()

# 2. Define the visualization function
def visualize_captum(token_ids, word_attributions):
    """Generates an HTML heatmap for IG attributions."""
    # Prepare token list (excluding special tokens like [CLS], [SEP], [PAD])
    tokens_to_display = tokenizer_xai.convert_ids_to_tokens(token_ids)
    
    # Filter out [CLS], [SEP], [PAD] and corresponding attributions for cleaner visualization
    filtered_tokens = []
    filtered_attributions = []
    for token, attr in zip(tokens_to_display, word_attributions):
        # Tokens like '##ing' are subwords. Keep them.
        if token not in ['[CLS]', '[SEP]', '[PAD]']:
            filtered_tokens.append(token)
            filtered_attributions.append(attr)

    print("\n--- Generating Visualization. Check the output directory for 'IG_Visualization.html' ---")

    # Use Captum's built-in visualization utility (often displays in notebook/prints HTML)
    # The output will color words: Red (Phishing support), Green (Safe support)
    viz.visualize_text(
        filtered_attributions,
        filtered_tokens,
        'Integrated Gradients Attributions',
        'Phishing Email',
        'IG_Visualization.html' # Saves the heatmap to this HTML file
    )

# Run the visualization
visualize_captum(token_ids, word_attributions)

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
from functools import partial
import numpy as np
import warnings
from captum.attr import IntegratedGradients
from IPython.display import HTML, display

warnings.filterwarnings('ignore')

# --- Custom Visualization Function ---
def plot_text_heatmap(tokens, attributions):
    """Generates an HTML representation of the text with attribution heatmap."""
    if not tokens or not attributions:
        return HTML("<p>No tokens or attributions to display.</p>")

    max_abs = np.max(np.abs(attributions))
    norm_attributions = attributions / max_abs if max_abs != 0 else np.zeros_like(attributions)

    html_output = '<p><strong>Integrated Gradients Heatmap (DistilBERT):</strong></p>'
    html_output += '<div style="line-height: 1.8; font-size: 14px; padding: 10px; border: 1px solid #ddd; border-radius: 5px; background-color: #f9f9f9;">'

    for token, score in zip(tokens, norm_attributions):
        intensity = min(1.0, abs(score) * 2.5) 
        if score > 0:
            style = f'background-color: rgba(0, 128, 0, {intensity:.2f}); color: white; padding: 1px 2px; border-radius: 3px; margin: 1px;'
        elif score < 0:
            style = f'background-color: rgba(255, 0, 0, {intensity:.2f}); color: white; padding: 1px 2px; border-radius: 3px; margin: 1px;'
        else:
            style = 'background-color: transparent; color: #333; padding: 1px 2px; margin: 1px;'

        # DistilBERT/BERT uses ## for subwords
        display_token = token.replace('##', '')
        html_output += f'<span style="{style}">{display_token}</span> '

    html_output += '</div>'
    return HTML(html_output)

# --- 1. Configuration and Model Loading (Changed to DistilBERT) ---
MODEL_NAME = 'distilbert-base-uncased' # Or your local path to fine-tuned DistilBERT
NUM_LABELS = 3
CLASS_NAMES = ['HAM', 'SPAM', 'AI-SPAM']

print(f"--- Initializing {MODEL_NAME} ---")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
    model.eval()
    print("Model and Tokenizer loaded successfully.")
except Exception as e:
    print(f"Error: {e}")

# --- 2. Predictor Function ---
def predictor(texts):
    with torch.no_grad():
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
        outputs = model(**inputs)
        probabilities = F.softmax(outputs.logits, dim=1).cpu().numpy()
    return probabilities

# --- 3. Example Data ---
example_ai_spam_text = (
    "URGENT NOTICE: Your Retail Rewards account balance has been suspended. "
    "Please confirm your identity via the dedicated Retail Secure Link."
)
example_label_index = 2 

# --- 4. LIME Analysis ---
print("\n--- 4. LIME Analysis ---")
explainer_lime = LimeTextExplainer(class_names=CLASS_NAMES)
explanation = explainer_lime.explain_instance(
    example_ai_spam_text, predictor, labels=[example_label_index], num_samples=1000
)
for feature, weight in explanation.as_list(label=example_label_index)[:10]:
    print(f"  Token: '{feature}' | Weight: {weight:.4f}")

# --- 5. Integrated Gradients (Updated for DistilBERT Structure) ---
print("\n--- 5. Integrated Gradients Analysis ---")

def forward_func_embeds(inputs_embeds, attention_mask=None):
    # DISTILBERT CHANGE: model.distilbert instead of model.electra
    outputs = model.distilbert(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
    hidden_state = outputs[0]  # (bs, seq_len, dim)
    # DistilBERT Classification head uses the first token ([CLS]) hidden state
    pooled_output = hidden_state[:, 0] 
    logits = model.classifier(pooled_output)
    return logits

# Prepare inputs
encoding = tokenizer.encode_plus(
    example_ai_spam_text, return_tensors='pt', padding='max_length', truncation=True, max_length=128
)
input_ids = encoding['input_ids']
attention_mask = encoding['attention_mask']

# DISTILBERT CHANGE: Accessing embeddings via model.distilbert.embeddings
input_embeddings = model.distilbert.embeddings(input_ids).requires_grad_()

# Baseline (Padding)
pad_token_id = tokenizer.pad_token_id
pad_embedding = model.distilbert.embeddings.word_embeddings(torch.tensor([pad_token_id]))
reference_embeddings = pad_embedding.unsqueeze(0).expand(input_embeddings.shape)

# Compute IG
ig = IntegratedGradients(forward_func_embeds)
attributions_ig, delta = ig.attribute(
    inputs=input_embeddings,
    baselines=reference_embeddings,
    target=example_label_index,
    n_steps=50,
    additional_forward_args=(attention_mask,),
    return_convergence_delta=True
)

# Process Results
all_tokens = tokenizer.convert_ids_to_tokens(input_ids.flatten())
attributions_ig_sum = attributions_ig.sum(dim=-1).squeeze(0).cpu().detach().numpy()

# Filter for display
ig_results = [(t, s) for t, s in zip(all_tokens, attributions_ig_sum) 
              if t not in ['[CLS]', '[SEP]', '[PAD]'] and not t.startswith('##')]
ig_results.sort(key=lambda x: abs(x[1]), reverse=True)

print(f"Top Tokens (IG):")
for token, score in ig_results[:10]:
    print(f"  Token: '{token}' | Score: {score:.4f}")

# Visualization
start_idx = 1
end_idx = list(input_ids.flatten().numpy()).index(tokenizer.sep_token_id)
display(plot_text_heatmap(all_tokens[start_idx:end_idx], attributions_ig_sum[start_idx:end_idx]))

print("\n--- XAI Analysis Complete ---")